# Nonlinear GRU — do the editability findings survive a nonlinear read-in and read-out?

**Thread:** `editability/` · **Registry:** `NONLINEAR_GRU_RUNS.md` (this directory) · **Metric/editor definitions:**
`../METRICS_AND_EDITORS.md` · **Conventions:** `../../../../CLAUDE.md`.
**Dataset:** `datasets/4_fixed_refl_inview` — `edits` split for every §3–§5 number, `test` split for probes and
predictive quality. **No retraining in this notebook**; the four checkpoints are trained by
`scripts/train_gru.py` with the recipe pinned in the registry.

## Why this notebook exists

Every editability result in this repo was established on a GRU whose **encoder is one `nn.Linear` + ReLU** and
whose **decoder is one bare `nn.Linear`**. A single-Linear decoder is **affine** in `h`, and that has a
consequence which invalidates one specific class of result:

$$\text{decode}(h_0 + d_1 + d_2) \;=\; \text{decode}(h_0+d_1) + \text{decode}(h_0+d_2) - \text{decode}(h_0)$$

holds **identically, for any vectors** $d_1, d_2$ — no structure in the latent required. Measured on the trained
baseline in `../delta_h_analysis.ipynb` §7b, the gap between the two sides is `6.6e-08`, i.e. machine precision.
So that notebook's "**edits superpose across objects**" result, *as read off the decoded observation*, was forced
by the decoder rather than earned by the representation. (Its **state-space** readouts were unaffected and stand.)

That raises the obvious worry about everything else: **how much of the editability picture is a property of
implicit recurrent world models, and how much is an artifact of a shallow read-in / read-out path?**

This notebook trains that worry out. It re-derives the thread's main findings on GRU variants with genuine MLP
encoders and decoders, against the exact baseline checkpoint the findings were established on:

1. **§1** — is the nonlinear model actually a comparable world model? (Nothing below is interpretable otherwise.)
2. **§2** — is position still **linearly** decodable from `h`, or did encoder depth move the information into a
   nonlinear code?
3. **§3** — does **editability still fail**: do structural editors stay inert while oracles succeed, on the same
   model, decoder and rollout?
4. **§4** — is a successful edit's direction still **orthogonal to the probe's row space** (at chance)?
5. **§5** — the **superposition** test, now with the affine identity broken by construction, plus the control
   that separates a real result from a forced one.

Each of §3 and §5 ships an observation-space waterfall of the same arms it tabulates.

## Definitions — read this before any number

### The models compared (rows copied from `NONLINEAR_GRU_RUNS.md`)

All share dataset `4_fixed_refl_inview` and an **identical recipe**: 400 epochs, batch 256, AdamW lr 1e-3,
weight decay 1e-4, `hidden_size` as stated, `num_layers=1`, dropout 0, next-frame MSE teacher forcing, no state
supervision. Depth `k` inserts `k × (Linear(H,H) + ReLU)`: encoder blocks go **after** the encoder `Linear+ReLU`,
decoder blocks **before** the decoder `Linear`.

| label used in every figure | checkpoint | enc depth | dec depth | `decode` | seed | params | role |
|---|---|---|---|---|---|---|---|
| `linear enc+dec · H256 · seed 0` | `runs/controls/H256` | 0 | 0 | **affine** | 0 | 460,672 | **baseline** — the exact checkpoint the editability findings and `delta_h_analysis` were established on |
| `nonlinear enc+dec · H256 · seed 0` | `runs/nonlinear_gru/NL_enc2dec2_s0` | 2 | 2 | nonlinear | 0 | 723,840 | **the main variant** |
| `nonlinear dec only · H256 · seed 0` | `runs/nonlinear_gru/NL_dec2_s0` | 0 | 2 | nonlinear | 0 | 592,256 | isolates **which half matters** — the affine artifact is a *decoder* property |
| `nonlinear enc+dec · H256 · seed 1` | `runs/nonlinear_gru/NL_enc2dec2_s1` | 2 | 2 | nonlinear | 1 | 723,840 | **seed control** for the main variant |
| `linear enc+dec · H512 · seed 0` | `runs/controls/H512` | 0 | 0 | affine | 0 | 1,704,064 | **capacity reference** — more parameters than either nonlinear variant, so a changed finding cannot be blamed on capacity alone |

**Suffix key.** `NL` = nonlinear · `enc2dec2` = 2 extra encoder + 2 extra decoder blocks · `dec2` = decoder depth
only · `s0`/`s1` = training seed.

### The states (all at the edit frame `ef = 20`)

| symbol | how it is built | what it represents |
|---|---|---|
| `h0` | teacher-force the **real, noisy** observations `obs[0..ef−1]` | the pre-edit state: the model's belief about frame `ef` *without* having seen the teleport |
| `h_pinv` | `h0 + A⁺(target − (A h0 + b))` | **readout injection** — the canonical failing structural editor |
| `h_dgrad` | Adam on `h` to match the **clean GT edit-frame observation** through this model's own decoder | **decoder gradient (oracle)** — has GT observation access |
| `h_ft` | from `h0`, teacher-force **N = 8 noise-matched rendered frames** interpolating the edited object pre-edit → target (other object held at its `ef` position) | **freeze-time (oracle)** |
| `h_cf` | teacher-force **clean renders of a counterfactual history** `0..ef−1` in which the edited object travelled at constant velocity *through* the target, the other object on its true path | **counterfactual state overwrite (oracle)** |

### The metrics — names, formulas, units, better-direction

Copied verbatim from `../METRICS_AND_EDITORS.md`; §4 formulas are **imported** from
`scripts/editability_metrics.py`, never re-derived here.

**Two ground-truth worlds.** `gt_edited` = `edits.clean_obs[ef]`, the world where the teleport happened.
`gt_unedited` = the counterfactual where it did not (edited object continues from `ef−1` along its own velocity,
other object at its true `ef` position). **Ray zones:** `target` = rays the edited object occupies in `gt_edited`;
`ghost` = rays it vacates; `collateral` = the other object's rays; `differing` = where the two worlds differ
(`|gt_edited − gt_unedited| > 1e-3`), the support of the Edit Index.

| metric | formula | units | better |
|---|---|---|---|
| **next-step RMSE vs clean** | `RMSE(pred_t, clean_obs[t+1])`, teacher-forced over the test split | obs intensity | ↓ |
| **open-loop RMSE vs clean @ K** | free-run K steps from a warmed state; `mean_s RMSE(roll_s, clean_obs[t+s])` | obs intensity | ↓ |
| **sharpness (TV ratio)** | total variation along the ray axis of the rollout ÷ that of the GT render | ratio | → 1 — **`< 1` = blurry mean-hedging, `> 1` = extra texture** (residual observation noise carried into the rollout: the models are trained on noisy frames but scored against the clean render) |
| **position R² (linear)** | `1 − ‖Y − (Ah+b)‖²/‖Y − Ȳ‖²`, least squares, `Y` = 2 objects × (x,y) | — | ↑ |
| **position R² (MLP)** | same, probe = 2-hidden-layer MLP (256 units, ReLU), 400 Adam steps | — | ↑ |
| **Target / Ghost / Collateral / Edit-frame RMSE** | `RMSE(edited₀, gt_edited)` over that zone, at rollout **step 0** (which decodes frame `ef`) | obs intensity | ↓ |
| **GT-traj RMSE** | `mean_s RMSE(edited_s, clean_obs[ef+s])` over the K-step rollout | obs intensity | ↓ |
| **fidelity ratio** | `GT-traj RMSE(editor) / GT-traj RMSE(unsteered)` | ratio | ↓ — **> 1 means the edit left the rollout *further* from the true post-edit world than doing nothing** |
| **Edit Index** | `(d_uned − d_edit)/(d_uned + d_edit)`, `d_· = RMSE(edited₀, gt_·)` over **differing** rays; per sample then averaged | −1…+1 | ↑ |
| **row-space fraction `f`** | `‖P_row(A)·Δh‖ / ‖Δh‖`, `P_row = A⁺A`; per sample then averaged | fraction | — (see chance) |
| **cos(composed, direct)** | `⟨u,v⟩/(‖u‖‖v‖)` between the composed and directly-built Δh, per sample then averaged | — | ↑ |

### Three baselines you must read these against

1. **The Edit Index has a per-model `−1` end.** A *perfect* predictor scores exactly −1 unsteered; a real one
   falls short by its own blur, because `d_unedited` is its one-step prediction error rather than 0. **The
   unsteered row appears in every table** and every index is read against its own model's row. The `+1` end is
   not shifted.
2. **The row-space fraction has a chance level of `√(d/H)`** — a random vector already has that share of its norm
   in a `d`-dimensional subspace, purely by dimension counting (0.125 for the `d=4` position probe at `H=256`).
   `f = 0.15` is *not* "15% aligned, mostly missing"; it is barely above chance. Because `H512` has a **different**
   `H`, and therefore a different chance level, the figure plots the **enrichment** `f / chance`; raw fractions are
   comparable only among the four `H=256` runs.
3. **A cosine is not a correlation.** cos 0.87 is a **29° angle**, and two equal-length vectors 29° apart differ by
   `2·sin(θ/2) ≈ 0.50` of their length. Angles are reported beside every cosine, and the mean cosine of *random*
   vectors is **0** (`1/√H` is the per-pair standard deviation, not a floor) — so every cosine below also carries
   an **empirical shuffled-pair floor**.

### Implementation details a reader would ask about

- `ef = 20` (the dataset's edit frame) · `K_ROLL = 15` rollout steps · `N_FT = 8` freeze-time frames ·
  `N_EVAL = 256` held-out edit samples for §3–§4 · `COMP_N = 96` for §5.
- **Every observation-space error is scored against the CLEAN render** (`clean_obs`), never the noisy `obs`.
  Teacher-forced *inputs* are noise-matched to training (`obs_noise_std = 0.2`) to stay in distribution.
- **±1 decode convention.** These are predict-next GRUs: `decode(h_t) ≈ obs[t+1]`. Warming to `ef` teacher-forces
  `obs[0..ef−1]`, so the rollout's **step 0 is sim frame `ef`** — `ROLL[:,0] ↔ clean_obs[ef]`. Cell [1] verifies
  this by measurement rather than asserting it.

In [ ]:
# [1] Setup: load the five checkpoints + the dataset, and verify the ±1 alignment BY MEASUREMENT.
import os, sys, json, time
sys.path.insert(0, "../../../..")
sys.path.insert(0, "../../../../scripts")
import numpy as np, torch, h5py
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from IPython.display import display, Markdown

from pim.world_models import load_checkpoint, load_dataset
from pim.simulator.sim import Scene, SimConfig
from pim.simulator.renderer import render_scene
from pim.figures.theme import style_ax
from editability_metrics import build_edit_zones, edit_scorecard, fidelity_ratio, _index_from

torch.manual_seed(0); np.random.seed(0)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
N_OBJ, K_ROLL, N_FT = 2, 15, 8
N_EVAL, COMP_N = 256, 96
OUT = "figures"; os.makedirs(OUT, exist_ok=True)

ROOT = "../../../.."
CKPT = {
    "linear enc+dec · H256 · seed 0":     f"{ROOT}/runs/controls/H256/best_model.pt",
    "nonlinear enc+dec · H256 · seed 0":  f"{ROOT}/runs/nonlinear_gru/NL_enc2dec2_s0/best_model.pt",
    "nonlinear dec only · H256 · seed 0": f"{ROOT}/runs/nonlinear_gru/NL_dec2_s0/best_model.pt",
    "nonlinear enc+dec · H256 · seed 1":  f"{ROOT}/runs/nonlinear_gru/NL_enc2dec2_s1/best_model.pt",
    "linear enc+dec · H512 · seed 0":     f"{ROOT}/runs/controls/H512/best_model.pt",
}
# Okabe-Ito, one colour per model, fixed for the whole notebook.
CLR = {"linear enc+dec · H256 · seed 0":     "#0072B2",
       "nonlinear enc+dec · H256 · seed 0":  "#D55E00",
       "nonlinear dec only · H256 · seed 0": "#E69F00",
       "nonlinear enc+dec · H256 · seed 1":  "#CC79A7",
       "linear enc+dec · H512 · seed 0":     "#56B4E9"}
BASE_M = "linear enc+dec · H256 · seed 0"          # the reference the findings were established on
MAIN_M = "nonlinear enc+dec · H256 · seed 0"       # the variant this notebook is about

MODELS, INFO = {}, {}
for name, p in CKPT.items():
    MODELS[name], INFO[name] = load_checkpoint(p, device=DEVICE)

bundle = load_dataset(f"{ROOT}/datasets/4_fixed_refl_inview", n_obj_keep=N_OBJ)
edits, test = bundle.edits, bundle.test
ef = edits.edit_frame; sim = test.config["dataset"]["sim"]; R = edits.obs_res
with h5py.File(edits.h5_path, "r") as f:
    VEL_ALL = f["velocities"][:, :, :N_OBJ, :].astype(np.float32)

@torch.no_grad()
def warm(model, obs_np, upto):
    """Teacher-force obs[0..upto-1]; returns the flat state, ALIGNED so decode(state) ↔ frame `upto`."""
    o = torch.from_numpy(obs_np).float().to(DEVICE); state = None
    for t in range(upto):
        _, state = model.step(o[:, t], state)
    return model.flat_state(state)

@torch.no_grad()
def roll(model, h_flat, steps=K_ROLL):
    """Free-run from a flat state. out[:, 0] = decode(h) = sim frame `ef`."""
    st = model.state_from_flat(h_flat); out = [model.decode(st)]
    for _ in range(steps - 1):
        p, st = model.predict_step(st); out.append(p)
    return torch.stack(out, 1).cpu().numpy()

# ── alignment CHECK on ORDINARY (non-edit) test sequences ──────────────────────
# Must NOT use the edits split: there the pre-edit state legitimately fails to predict frame `ef`
# (the teleport is exactly what it has not seen), which would confound the very effect we study.
print("alignment check — RMSE(decode(warm to t), clean_obs[t+k]); the minimum must sit at k=0:")
for name, m in MODELS.items():
    hc = warm(m, test.obs[:256].astype(np.float32), ef)
    dc = m.decode(m.state_from_flat(hc)).cpu().numpy()
    errs = {k: float(np.sqrt(((dc - test.clean_obs[:256, ef + k]) ** 2).mean())) for k in (-1, 0, 1)}
    best = min(errs, key=errs.get)
    print(f"  {name:<36s} k=-1 {errs[-1]:.4f} | k=0 {errs[0]:.4f} | k=+1 {errs[1]:.4f}"
          f"  -> min at k={best} {'PASS' if best == 0 else 'FAIL'}")

obs_ev = edits.obs[:N_EVAL].astype(np.float32)
H0 = {name: warm(m, obs_ev, ef) for name, m in MODELS.items()}
print(f"\nedits split: analysis on samples [0,{N_EVAL}) | ef={ef} | K_ROLL={K_ROLL}")

In [ ]:
# [2] Table 1 — the architectural fact this notebook turns on: is `decode` affine in h?
# Affine means decode(h0+d1+d2) - [decode(h0+d1)+decode(h0+d2)-decode(h0)] == 0 for ANY d1, d2.
# We measure it on RANDOM directions (the identity is meant to hold for any vector, not just real states)
# AND on real state displacements, at each model's own scale.
rows = ["| model | enc depth | dec depth | params | `has_affine_decoder` | affine violation, random directions | affine violation, real Δh | reference: RMS of a decoded observation |",
        "|---|---|---|---|---|---|---|---|"]
AFFINE = {}
for name, m in MODELS.items():
    cfg = m.cfg
    g = torch.Generator(device="cpu").manual_seed(1)
    h_real = H0[name]
    scale = float(torch.linalg.norm(h_real, dim=-1).mean())
    d1, d2 = (torch.randn(64, m.hidden_size, generator=g).to(DEVICE) for _ in range(2))
    d1 = d1 / d1.norm(dim=-1, keepdim=True) * scale; d2 = d2 / d2.norm(dim=-1, keepdim=True) * scale
    s = m.state_from_flat
    with torch.no_grad():
        h0r = h_real[:64]
        v_rand = float(torch.sqrt(((m.decode(s(h0r + d1 + d2))
                                    - (m.decode(s(h0r + d1)) + m.decode(s(h0r + d2)) - m.decode(s(h0r)))) ** 2).mean()))
        # real displacements: use two other samples' states as the "edits"
        e1, e2 = h_real[64:128] - h_real[:64], h_real[128:192] - h_real[:64]
        v_real = float(torch.sqrt(((m.decode(s(h0r + e1 + e2))
                                    - (m.decode(s(h0r + e1)) + m.decode(s(h0r + e2)) - m.decode(s(h0r)))) ** 2).mean()))
        obs_rms = float(torch.sqrt((m.decode(s(h_real)) ** 2).mean()))
    AFFINE[name] = dict(rand=v_rand, real=v_real, obs_rms=obs_rms)
    rows.append(f"| {name} | {getattr(cfg,'enc_hidden_layers',0)} | {getattr(cfg,'dec_hidden_layers',0)} | "
                f"{sum(p.numel() for p in m.parameters()):,} | {m.has_affine_decoder} | "
                f"{v_rand:.2e} | {v_real:.2e} | {obs_rms:.3f} |")
display(Markdown(
    "**Table 1 — is the decoder affine?** Violation = RMSE between `decode(h0+d1+d2)` and "
    "`decode(h0+d1)+decode(h0+d2)−decode(h0)`, in observation-intensity units, with the RMS of a decoded "
    "observation given as the reference scale. Displacements are scaled to the model's own mean ‖h‖. A value at "
    "machine precision means the identity holds **by algebra**, so any 'edits superpose' result read off the "
    "decoded observation is forced rather than earned.\n\n" + "\n".join(rows)))

---
## §1 — Is the nonlinear model a comparable world model?

Nothing in §2–§5 is interpretable if the variants are worse predictors: a degraded model would score badly on
editability for reasons that have nothing to do with the architecture question, and the Edit Index's `−1` end
would move (it sits at each model's own next-step error). This section establishes the common ground.

Three quantities: teacher-forced **next-step RMSE vs clean**, free-running **open-loop RMSE vs clean** over the
same K-step horizon the editors are scored on, and **sharpness** (rollout total variation ÷ GT total variation,
where `< 1` means the model is hedging toward a blurry mean). The **noise floor** line is `RMSE(obs, clean_obs)`
= the error of simply echoing the noisy input; it is a *reference scale*, not a bound — a recurrent model that
denoises across frames legitimately scores below it.

In [ ]:
# [3] Table 2 + Fig 1 — predictive quality. Everything scored against the CLEAN render.
NPQ = 600
obs_pq   = test.obs[:NPQ].astype(np.float32)
clean_pq = test.clean_obs[:NPQ].astype(np.float32)
noise_floor = float(np.sqrt(((obs_pq - clean_pq) ** 2).mean()))

def total_variation(x):
    return np.abs(np.diff(x, axis=-1)).mean()

PQ = {}
for name, m in MODELS.items():
    o = torch.from_numpy(obs_pq).to(DEVICE)
    with torch.no_grad():
        pred, _ = m.observe_sequence(o)                       # (N, T-1, R); pred[:, t] ↔ frame t+1
    ns = float(np.sqrt(((pred.cpu().numpy() - clean_pq[:, 1:]) ** 2).mean()))
    h_w = warm(m, obs_pq, ef)
    rl = roll(m, h_w, K_ROLL)                                 # rl[:, 0] ↔ frame ef
    gt = clean_pq[:, ef:ef + K_ROLL]
    ol = float(np.sqrt(((rl - gt) ** 2).mean()))
    ol_by_step = np.sqrt(((rl - gt) ** 2).mean(axis=(0, 2)))
    PQ[name] = dict(next_step=ns, open_loop=ol, ol_by_step=ol_by_step,
                    tv_ratio=total_variation(rl) / total_variation(gt),
                    val_loss=INFO[name].val_loss, epoch=INFO[name].epoch)

rows = ["| model | val loss (training, vs noisy) | next-step RMSE vs clean ↓ | open-loop RMSE vs clean, K=15 ↓ | sharpness (TV ratio) → 1 |",
        "|---|---|---|---|---|"]
for name in MODELS:
    q = PQ[name]
    rows.append(f"| {name} | {q['val_loss']:.5f} | **{q['next_step']:.4f}** | {q['open_loop']:.4f} | {q['tv_ratio']:.3f} |")
display(Markdown(f"**Table 2 — predictive quality.** N={NPQ} held-out `test` sequences. Reference scale: the "
                 f"noise floor `RMSE(obs, clean_obs)` = **{noise_floor:.4f}** (the error of echoing the noisy "
                 f"input). Val loss is the training objective (MSE against **noisy** targets) and is shown only "
                 f"for provenance — it is not comparable to the clean-referenced RMSE columns.\n\n" + "\n".join(rows)))

plt.style.use("default")
fig, ax = plt.subplots(1, 3, figsize=(17.5, 4.4))
names = list(MODELS)
cats = ["next-step RMSE\nvs clean", "open-loop RMSE\nvs clean (K=15)"]
xi = np.arange(len(cats)); w = 0.15
for k, name in enumerate(names):
    ax[0].bar(xi + (k - (len(names)-1)/2) * w, [PQ[name]["next_step"], PQ[name]["open_loop"]], w,
              color=CLR[name], label=name)
ax[0].axhline(noise_floor, color="0.35", ls=":", lw=1.3)
ax[0].annotate(f"noise floor {noise_floor:.3f}", xy=(len(cats)-0.45, noise_floor), fontsize=7.5,
               color="0.35", ha="right", va="bottom")
ax[0].set_xticks(xi); ax[0].set_xticklabels(cats, fontsize=8.5)
ax[0].set_ylabel("RMSE vs clean render (obs intensity)")
ax[0].set_title("(a) predictive error — lower is better", fontsize=10); ax[0].grid(alpha=0.3, axis="y"); style_ax(ax[0])

for name in names:
    ax[1].plot(np.arange(K_ROLL), PQ[name]["ol_by_step"], color=CLR[name], lw=2.0, label=name)
ax[1].set_xlabel("rollout step s (0 = sim frame ef)"); ax[1].set_ylabel("RMSE vs clean render")
ax[1].set_title("(b) open-loop error by step", fontsize=10); ax[1].grid(alpha=0.3); style_ax(ax[1])

for k, name in enumerate(names):
    ax[2].bar(k, PQ[name]["tv_ratio"], 0.62, color=CLR[name])
ax[2].axhline(1.0, color="0.35", ls=":", lw=1.3)
ax[2].annotate("as sharp as the GT render", xy=(len(names)-0.4, 1.0), fontsize=7.5, color="0.35",
               ha="right", va="bottom")
ax[2].set_xticks([]); ax[2].set_ylabel("rollout TV ÷ GT TV")
ax[2].set_title("(c) sharpness: 1 = as sharp as GT\nbelow 1 = blurry, above 1 = extra texture", fontsize=10)
ax[2].grid(alpha=0.3, axis="y"); style_ax(ax[2])

fig.legend(handles=[Line2D([0],[0], color=CLR[n], lw=6, label=n) for n in names],
           loc="upper center", ncol=3, fontsize=8.5, frameon=False, bbox_to_anchor=(0.5, 1.10))
fig.suptitle("Fig 1 — predictive quality of every model, on held-out test sequences", y=1.16, fontsize=12)
fig.tight_layout(); fig.savefig(f"{OUT}/fig1_predictive_quality.png", dpi=130, bbox_inches="tight")
display(fig); plt.close(fig)

---
## §2 — Is position still *linearly* decodable from `h`?

The established finding is "**readable ≠ controllable**": position is linearly readable at R² ≈ 0.84 and yet no
structural editor can move it. A nonlinear encoder is the obvious threat to the first half — it could push the
positional information into a code that only a nonlinear probe can read, which would change what §4's row-space
measurement even means (the row space of a *linear* probe is only the reachable set of a *linear* injection).

So both probes are fit on every model: **linear** (least squares) and **MLP** (2 hidden layers, 256 units, ReLU,
400 full-batch Adam steps). The gap between them is the quantity of interest, not either alone.

Probes are fit on **aligned** states — the same alignment as the states we edit, so `decode(bank[:, t]) ↔ frame
t+1`. Fitting on one alignment and applying on another is a one-frame mismatch that a Δh study cannot afford.

In [ ]:
# [4] Table 3 + Fig 2 — linear vs MLP readability of position and velocity.
@torch.no_grad()
def aligned_bank(model, obs_np):
    """States aligned exactly like the ones we edit: decode(bank[:, t]) ↔ sim frame t+1."""
    o = torch.from_numpy(obs_np).float().to(DEVICE)
    T = o.shape[1]; state = None; out = []
    for t in range(T - 1):
        _, state = model.step(o[:, t], state)
        out.append(model.flat_state(state))
    return torch.stack(out, 1)

# The 80/20 split is by SEQUENCE (the flatten is over (N, T), so the first 80% of rows are the
# first 80% of sequences) — a random row split would leak neighbouring frames across the boundary.
# BOTH probes are fit on the same 80% and scored on the same held-out 20%: an in-sample linear R²
# against a held-out MLP R² is not a like-for-like comparison, and the MLP-minus-linear gap is the
# whole point of this table.
HOLDOUT = 0.2

def _xy(model, obs_np, y_np):
    Hs = aligned_bank(model, obs_np).cpu().numpy()
    T = Hs.shape[1]
    X = Hs.reshape(-1, Hs.shape[-1]); Y = y_np[:, 1:1 + T].reshape(-1, y_np.shape[-1])
    return X, Y, int((1 - HOLDOUT) * len(X))

def _r2(pred, Y):
    return float(1 - ((pred - Y) ** 2).sum() / ((Y - Y.mean(0)) ** 2).sum())

def fit_linear_probe(model, obs_np, y_np):
    """Least-squares probe h -> y, fit on the 80%, scored on the held-out 20%."""
    X, Y, n_tr = _xy(model, obs_np, y_np)
    Aug = np.concatenate([X, np.ones((len(X), 1), np.float32)], 1)
    sol, *_ = np.linalg.lstsq(Aug[:n_tr], Y[:n_tr], rcond=None)
    A = sol[:-1].T.astype(np.float32); b = sol[-1].astype(np.float32)
    pred_ho = X[n_tr:] @ sol[:-1] + sol[-1]
    A_t = torch.tensor(A, device=DEVICE)
    A_pinv = torch.tensor(np.linalg.pinv(A), device=DEVICE)
    return dict(A=A_t, b=torch.tensor(b, device=DEVICE), A_pinv=A_pinv, P_row=A_pinv @ A_t,
                rmse=float(np.sqrt(((pred_ho - Y[n_tr:]) ** 2).mean())),
                r2=_r2(pred_ho, Y[n_tr:]),
                r2_insample=_r2(X[:n_tr] @ sol[:-1] + sol[-1], Y[:n_tr]), d=A.shape[0])

def fit_mlp_probe(model, obs_np, y_np, steps=400, hidden=256, seed=0):
    """MLP probe h -> y on the SAME split, so the gap to the linear probe is like-for-like."""
    X, Y, n_tr = _xy(model, obs_np, y_np)
    Xt = torch.tensor(X, device=DEVICE); Yt = torch.tensor(Y, device=DEVICE)
    torch.manual_seed(seed)
    net = torch.nn.Sequential(torch.nn.Linear(X.shape[1], hidden), torch.nn.ReLU(),
                              torch.nn.Linear(hidden, hidden), torch.nn.ReLU(),
                              torch.nn.Linear(hidden, Y.shape[1])).to(DEVICE)
    opt = torch.optim.Adam(net.parameters(), lr=1e-3)
    for _ in range(steps):
        opt.zero_grad(); loss = ((net(Xt[:n_tr]) - Yt[:n_tr]) ** 2).mean(); loss.backward(); opt.step()
    with torch.no_grad():                                    # scored on the HELD-OUT 20%
        p = net(Xt[n_tr:]).cpu().numpy(); yv = Y[n_tr:]
        rmse = float(np.sqrt(((p - yv) ** 2).mean()))
    return dict(r2=_r2(p, yv), rmse=rmse, d=Y.shape[1])

NPROBE = 600
obs_pr = test.obs[:NPROBE].astype(np.float32)
pos_pr = test.positions[:NPROBE, :, :N_OBJ, :].reshape(NPROBE, -1, N_OBJ * 2)
with h5py.File(test.h5_path, "r") as f:
    vel_pr = f["velocities"][:NPROBE, :, :N_OBJ, :].astype(np.float32).reshape(NPROBE, -1, N_OBJ * 2)
posvel_pr = np.concatenate([pos_pr, vel_pr], -1)

PROBES, MLPP = {}, {}
t0 = time.perf_counter()
for name, m in MODELS.items():
    PROBES[name] = {"position (d=4)": fit_linear_probe(m, obs_pr, pos_pr),
                    "position+velocity (d=8)": fit_linear_probe(m, obs_pr, posvel_pr)}
    MLPP[name] = {"position (d=4)": fit_mlp_probe(m, obs_pr, pos_pr),
                  "velocity (d=4)": fit_mlp_probe(m, obs_pr, vel_pr)}
    MLPP[name]["velocity (d=4) linear"] = fit_linear_probe(m, obs_pr, vel_pr)
print(f"fitted probes for {len(MODELS)} models in {time.perf_counter()-t0:.0f}s")

rows = ["| model | position R² (linear) ↑ | position R² (MLP) ↑ | MLP − linear gap | velocity R² (linear) ↑ | velocity R² (MLP) ↑ | position RMSE (linear, sim units) ↓ | linear in-sample R² (overfit check) |",
        "|---|---|---|---|---|---|---|---|"]
for name in MODELS:
    lp = PROBES[name]["position (d=4)"]; mp = MLPP[name]["position (d=4)"]
    lv = MLPP[name]["velocity (d=4) linear"]; mv = MLPP[name]["velocity (d=4)"]
    rows.append(f"| {name} | **{lp['r2']:.3f}** | {mp['r2']:.3f} | {mp['r2']-lp['r2']:+.3f} | "
                f"{lv['r2']:.3f} | {mv['r2']:.3f} | {lp['rmse']:.3f} | {lp['r2_insample']:.3f} |")
display(Markdown("**Table 3 — how readable is the physical state, linearly and nonlinearly?** **Both** probes are "
                 "fit on the same 80% of sequences and scored on the same held-out 20%, so the comparison is "
                 "like-for-like. The last column is the linear probe's in-sample R², shown only to confirm it is "
                 "not overfitting. "
                 "The **MLP − linear gap** is the quantity of interest: it is how much of the positional "
                 "information sits in a nonlinearly-coded form.\n\n" + "\n".join(rows)))

fig, ax = plt.subplots(1, 2, figsize=(14.5, 4.4))
names = list(MODELS)
for j, (quant, keys) in enumerate([("position", ("position (d=4)", "position (d=4)")),
                                   ("velocity", ("velocity (d=4) linear", "velocity (d=4)"))]):
    a = ax[j]; xi = np.arange(2); w = 0.15
    for k, name in enumerate(names):
        lin = (PROBES[name][keys[0]]["r2"] if j == 0 else MLPP[name][keys[0]]["r2"])
        a.bar(xi + (k - (len(names)-1)/2) * w, [lin, MLPP[name][keys[1]]["r2"]], w, color=CLR[name], label=name)
    a.set_xticks(xi); a.set_xticklabels(["linear probe", "MLP probe"], fontsize=9)
    a.set_ylim(0, 1.02); a.set_ylabel("R²"); a.set_title(f"({'ab'[j]}) {quant} readability", fontsize=10)
    a.grid(alpha=0.3, axis="y"); style_ax(a)
fig.legend(handles=[Line2D([0],[0], color=CLR[n], lw=6, label=n) for n in names],
           loc="upper center", ncol=3, fontsize=8.5, frameon=False, bbox_to_anchor=(0.5, 1.12))
fig.suptitle("Fig 2 — is the physical state still linearly readable after adding encoder depth?", y=1.19, fontsize=12)
fig.tight_layout(); fig.savefig(f"{OUT}/fig2_readability.png", dpi=130, bbox_inches="tight")
display(fig); plt.close(fig)

---
## §3 — Does editability still fail?

The central negative of the thread, restated as a bracket that runs **on one model, one decoder, one rollout**:

- **Structural editors** (readout injection) write to `h` directly using only what a probe exposes.
- **Oracles** (decoder gradient, freeze-time, counterfactual overwrite) get ground-truth access — the GT
  observation, or externally-rendered frames of the edited world.

If the oracles succeed while the structural editor is inert *on the same model*, the barrier cannot be "the model
is bad" or "the decoder can't render it"; it is the **reachability of the edit map**. That bracket is what this
section reproduces on the nonlinear variants.

One thing genuinely changes here. **Decoder gradient is a different optimisation problem** on a nonlinear
decoder: on the baseline it descends a convex quadratic in `h`, whereas here it descends a nonconvex objective
through an MLP. Whether it still reaches the target is not a foregone conclusion, and the table says so directly.

In [ ]:
# [5] Build the oracle edit states, the readout-injection edit, and the canonical ray zones.
REFL = np.array([sim["refl_min"], sim["refl_max"]], np.float32)
RAD  = np.array([sim["radius"]] * N_OBJ, np.float32)
COL  = np.tile(np.array([[1, 1, 1]], np.float32), (N_OBJ, 1))
DT   = float(sim["dt"]); OBS_NOISE = float(sim["obs_noise_std"])

def _cfg(nf, noise):
    return SimConfig(seed=0, y_near=sim["y_near"], y_far=sim["y_far"], x_near=sim["x_near"], x_far=sim["x_far"],
                     n_objects=N_OBJ, radius=sim["radius"], n_frames=nf, dt=sim["dt"], obs_res=sim["obs_res"],
                     refl_min=sim["refl_min"], refl_max=sim["refl_max"], fixed_reflectivities=True,
                     obs_noise_std=noise, boundary="open", always_in_frustum=False)

def render_traj(pos_seq, noise=0.0):
    _, _, inten = render_scene(Scene(positions=pos_seq, velocities=np.zeros_like(pos_seq), radii=RAD,
                                     colors=COL, reflectivities=REFL, config=_cfg(len(pos_seq), noise)))
    return inten.astype(np.float32)

def build_sequences(idx):
    """Counterfactual-history and freeze-time observation sequences for edit samples `idx`."""
    n = len(idx)
    oe  = edits.edit_object[idx].astype(int)
    pos = edits.positions[idx][:, :, :N_OBJ, :].astype(np.float32)
    tgt = pos[:, ef].copy()                                   # the edited object is already at the target
    pre = pos[:, ef - 1]
    cf_obs = np.zeros((n, ef, R), np.float32)
    ft_obs = np.zeros((n, N_FT, R), np.float32)
    t_idx = np.arange(ef)
    for i in range(n):
        o, other = oe[i], 1 - oe[i]
        v = VEL_ALL[idx[i], ef, o]
        cf = np.zeros((ef, N_OBJ, 2), np.float32)
        cf[:, o]     = tgt[i, o][None, :] - v[None, :] * (ef - t_idx)[:, None] * DT
        cf[:, other] = pos[i, :ef, other]
        cf_obs[i] = render_traj(cf)
        fr = np.zeros((N_FT, N_OBJ, 2), np.float32)
        for j in range(N_FT):
            fr[j, o] = pre[i, o] + ((j + 1) / N_FT) * (tgt[i, o] - pre[i, o]); fr[j, other] = tgt[i, other]
        ft_obs[i] = render_traj(fr, OBS_NOISE)
    return dict(cf=cf_obs, ft=ft_obs, oe=oe, tgt=tgt, pre=pre)

@torch.no_grad()
def continue_from(model, h_flat, frames):
    st = model.state_from_flat(h_flat)
    o = torch.from_numpy(frames).float().to(DEVICE)
    for t in range(frames.shape[1]):
        _, st = model.step(o[:, t], st)
    return model.flat_state(st)

def decoder_gradient(model, h_flat, target_obs, steps=400, lr=0.05):
    """ORACLE: Adam on h so that this model's OWN decoder renders the GT edit-frame observation.
    On an affine decoder this is a convex least-squares problem; on an MLP decoder it is not."""
    h = h_flat.clone().detach().requires_grad_(True)
    tgt = torch.from_numpy(target_obs).float().to(DEVICE)
    opt = torch.optim.Adam([h], lr=lr)
    for _ in range(steps):
        opt.zero_grad()
        ((model.decode(model.state_from_flat(h)) - tgt) ** 2).mean().backward()
        opt.step()
    return h.detach()

t0 = time.perf_counter()
IDX = np.arange(N_EVAL)
SEQ = build_sequences(IDX)
print(f"rendered {N_EVAL} counterfactual + freeze-time sequences in {time.perf_counter()-t0:.0f}s")

gt_edit_frame = edits.clean_obs[:N_EVAL, ef, :].astype(np.float32)   # the oracle's target: CLEAN render
tgt4 = torch.from_numpy(SEQ["tgt"].reshape(N_EVAL, N_OBJ * 2)).float().to(DEVICE)

STATES = {}
for name, m in MODELS.items():
    h0 = H0[name]
    P = PROBES[name]["position (d=4)"]
    STATES[name] = {
        "unsteered (no edit)":                     h0,
        "readout injection (structural)":          h0 + (tgt4 - (h0 @ P["A"].T + P["b"])) @ P["A_pinv"].T,
        "decoder gradient (oracle)":               decoder_gradient(m, h0, gt_edit_frame),
        "freeze-time teacher forcing (oracle)":    continue_from(m, h0, SEQ["ft"]),
        "counterfactual state overwrite (oracle)": warm(m, SEQ["cf"], ef),
    }
EDITORS = list(STATES[BASE_M])

gt_roll = edits.clean_obs[:N_EVAL, ef:ef + K_ROLL, :].astype(np.float32)
ZONES = build_edit_zones(pre_pos=SEQ["pre"], tgt_pos=SEQ["tgt"], pre_vel=VEL_ALL[IDX, ef - 1, :N_OBJ, :],
                         edit_object=SEQ["oe"], sim=sim, n_obj=N_OBJ,
                         traj_pos=edits.positions[:N_EVAL, ef:ef + K_ROLL, :N_OBJ, :].astype(np.float32),
                         gt_edited_traj=gt_roll)
print(f"built {len(EDITORS)} states x {len(MODELS)} models | ray zones/sample: "
      f"target {ZONES.target.sum(1).mean():.1f}, ghost {ZONES.ghost.sum(1).mean():.1f}, "
      f"differing {ZONES.differing.sum(1).mean():.1f}")

In [ ]:
# [6] Table 4 + Fig 3 — the canonical §4 scorecard for every editor on every model.
CARDS, ROLLS = {}, {}
for name, m in MODELS.items():
    CARDS[name], ROLLS[name] = {}, {}
    for lab in EDITORS:
        rl = roll(m, STATES[name][lab]); ROLLS[name][lab] = rl
        c = edit_scorecard(rl, ZONES, gt_roll)
        c["fidelity_ratio"] = 1.0 if lab == "unsteered (no edit)" else fidelity_ratio(c, CARDS[name]["unsteered (no edit)"])
        CARDS[name][lab] = c

rows = ["| model | editor | Edit Index (step 0) ↑ | Edit Index (step 14) ↑ | Target RMSE ↓ | Ghost RMSE ↓ | Collateral RMSE ↓ | GT-traj RMSE ↓ | fidelity ratio ↓ |",
        "|---|---|---|---|---|---|---|---|---|"]
for name in MODELS:
    for lab in EDITORS:
        c = CARDS[name][lab]
        rows.append(f"| {name} | {lab} | **{c['edit_index']:+.2f}** | {c['edit_index_by_step'][-1]:+.2f} | "
                    f"{c['target_rmse']:.3f} | {c['ghost_rmse']:.3f} | {c['collateral_rmse']:.3f} | "
                    f"{c['gt_traj_rmse']:.3f} | {c['fidelity_ratio']:.2f} |")
display(Markdown("**Table 4 — does the edit land, and does it hold?** Edit Index: +1 = the output *is* the world "
                 f"where the edit happened, −1 = the world where it did not, 0 = equidistant. N={N_EVAL} held-out "
                 "edits. **Read each index against its own model's `unsteered` row** — that is where this model's "
                 "−1 end actually sits. A fidelity ratio > 1 means the edited rollout ended *further* from the "
                 "true post-edit world than doing nothing.\n\n" + "\n".join(rows)))

fig, ax = plt.subplots(1, 3, figsize=(18.5, 5.0))
names = list(MODELS); xi = np.arange(len(EDITORS)); w = 0.15
for k, name in enumerate(names):
    ax[0].barh(xi + (k - (len(names)-1)/2) * w, [CARDS[name][l]["edit_index"] for l in EDITORS], w,
               color=CLR[name], label=name)
for x, lab in [(1.0, "edited world"), (0.0, "equidistant"), (-1.0, "unedited world")]:
    ax[0].axvline(x, color="0.4", ls=":", lw=1.0)
    ax[0].annotate(lab, xy=(x, len(EDITORS) - 0.45), fontsize=7, color="0.35", ha="center", va="top", rotation=90)
ax[0].set_yticks(xi); ax[0].set_yticklabels(EDITORS, fontsize=8)
ax[0].set_xlim(-1.05, 1.15); ax[0].set_xlabel("Edit Index at step 0")
ax[0].set_title("(a) does the edit land?", fontsize=10); ax[0].grid(alpha=0.3, axis="x"); style_ax(ax[0])

for k, name in enumerate(names):
    ax[1].barh(xi + (k - (len(names)-1)/2) * w, [CARDS[name][l]["ghost_rmse"] for l in EDITORS], w, color=CLR[name])
ax[1].set_yticks(xi); ax[1].set_yticklabels([]); ax[1].set_xlabel("Ghost RMSE vs clean GT (obs intensity)")
ax[1].set_title("(b) did the object LEAVE where it was?", fontsize=10); ax[1].grid(alpha=0.3, axis="x"); style_ax(ax[1])

for name in names:
    ax[2].plot(np.arange(K_ROLL), CARDS[name]["counterfactual state overwrite (oracle)"]["edit_index_by_step"],
               color=CLR[name], lw=2.0)
    ax[2].plot(np.arange(K_ROLL), CARDS[name]["readout injection (structural)"]["edit_index_by_step"],
               color=CLR[name], lw=1.6, ls="--")
ax[2].axhline(0, color="0.4", ls=":", lw=1.0); ax[2].set_ylim(-1.05, 1.05)
ax[2].set_xlabel("rollout step s (0 = sim frame ef)"); ax[2].set_ylabel("Edit Index")
ax[2].set_title("(c) does it HOLD? solid = counterfactual overwrite,\ndashed = readout injection", fontsize=10)
ax[2].grid(alpha=0.3); style_ax(ax[2])

fig.legend(handles=[Line2D([0],[0], color=CLR[n], lw=6, label=n) for n in names],
           loc="upper center", ncol=3, fontsize=8.5, frameon=False, bbox_to_anchor=(0.5, 1.08))
fig.suptitle("Fig 3 — the editability bracket: oracles versus a structural editor, on every model", y=1.14, fontsize=12)
fig.tight_layout(); fig.savefig(f"{OUT}/fig3_edit_index.png", dpi=130, bbox_inches="tight")
display(fig); plt.close(fig)

In [ ]:
# [7] Fig 4 — observation waterfalls for the §3 editors (canonical spec, one helper for every waterfall).
N_CTX = 6
DARK, TXT, TICK, EDIT_C = "#0a0a14", "#a3adc2", "#808a9d", "#fa8850"
TARGET_C, GHOST_C = "#00E676", "#FF5252"
CTX_EDITS = edits.obs[:N_EVAL, ef - N_CTX:ef, :].astype(np.float32)  # the NOISY frames actually teacher-forced

def _cx(mask):
    i = np.where(mask)[0]; return i.mean() if i.size else np.nan

def waterfall_grid(col_titles, col_bodies, samples, suptitle, fname,
                   tgt_lines, ghost_lines, ctx, ylab=None, dpi=115):
    """Canonical waterfall (CLAUDE.md fixed spec).

    gray cmap on dark bg; N_CTX NOISY context frames above a dashed edit-frame line; below it EVERY
    column shows its OWN free-run from step 0 (which decodes sim frame `ef`). There is deliberately NO
    shared teacher-forced `ef` row: only an oracle-observation editor ever sees that frame, and painting
    it into every column would hide the exact frame the §4 scorecard scores.
    `ctx`: (N, N_CTX, R) context frames, passed explicitly so no caller can inherit another
    figure's context by accident. `tgt_lines`/`ghost_lines`: lists of per-sample ray coordinates
    (one entry per object/locator)."""
    ncol = len(col_titles)
    fig, axes = plt.subplots(len(samples), ncol, figsize=(3.0 * ncol, 3.4 * len(samples)),
                             squeeze=False, facecolor=DARK)
    for r, smp in enumerate(samples):
        for c in range(ncol):
            ax = axes[r][c]; ax.set_facecolor(DARK)
            panel = np.clip(np.concatenate([ctx[smp], col_bodies[c][smp]], axis=0), 0, 1)
            ax.imshow(panel, aspect="auto", origin="upper", cmap="gray", vmin=0, vmax=1,
                      interpolation="nearest")
            for sp in ax.spines.values(): sp.set_edgecolor(TICK)
            ax.axhline(N_CTX - 0.5, color=EDIT_C, lw=1.4, ls="--", alpha=0.95)
            for tl in tgt_lines:
                if not np.isnan(tl[smp]): ax.axvline(tl[smp], color=TARGET_C, lw=1.5, alpha=0.9)
            for gl in ghost_lines:
                if not np.isnan(gl[smp]): ax.axvline(gl[smp], color=GHOST_C, ls="--", lw=1.5, alpha=0.9)
            if r == 0: ax.set_title(col_titles[c], fontsize=7.5, color=TXT)
            if c == 0:
                ax.set_ylabel((ylab or "sample {s}") .format(s=smp) + "\nsim frame", fontsize=8, color=TXT)
                ax.set_yticks([0, N_CTX, N_CTX + 7, N_CTX + 14])
                ax.set_yticklabels([ef - N_CTX, ef, ef + 7, ef + 14], fontsize=7)
            else: ax.set_yticks([])
            ax.set_xlabel("ray", fontsize=8, color=TXT); ax.tick_params(colors=TICK, labelsize=7)
    handles = [Line2D([0],[0], color=TARGET_C, lw=2.2, label="target location (where the object should be)"),
               Line2D([0],[0], color=GHOST_C, ls="--", lw=2.2, label="ghost location (where it was before the edit)"),
               Line2D([0],[0], color=EDIT_C, ls="--", lw=2.2,
                      label=f"edit applied here — {N_CTX} noisy context frames above; every row below is that "
                            f"column's OWN free-run, step 0 = sim frame {ef}")]
    fig.legend(handles=handles, loc="upper center", ncol=1, fontsize=8.5, frameon=False,
               labelcolor=TXT, bbox_to_anchor=(0.5, 0.965))
    fig.suptitle(suptitle, y=1.0, fontsize=10.5, color=TXT)
    fig.tight_layout(rect=[0, 0, 1, 0.885])
    fig.savefig(f"{OUT}/{fname}", dpi=dpi, bbox_inches="tight", facecolor=DARK)
    display(fig); plt.close(fig); print("saved", fname)

# samples: the largest teleports that also vacate a well-defined ghost region
teleport = np.linalg.norm(SEQ["tgt"][np.arange(N_EVAL), SEQ["oe"]] - SEQ["pre"][np.arange(N_EVAL), SEQ["oe"]], axis=-1)
SAMPLES = list(np.argsort(teleport * (ZONES.ghost.sum(1) >= 3))[::-1][:3])
tgt_cx = np.array([_cx(ZONES.target[i]) for i in range(N_EVAL)])
gho_cx = np.array([_cx(ZONES.ghost[i]) for i in range(N_EVAL)])
print("waterfall samples (largest teleports):", SAMPLES)

for mdl in (BASE_M, MAIN_M):
    titles = ["GT (sim)\nthe true post-edit world"] + [
        f"{l}\nEdit Index {CARDS[mdl][l]['edit_index']:+.2f} · ghost {CARDS[mdl][l]['ghost_rmse']:.2f}"
        for l in EDITORS]
    bodies = [gt_roll] + [ROLLS[mdl][l] for l in EDITORS]
    waterfall_grid(titles, bodies, SAMPLES,
                   f"Fig 4{'a' if mdl == BASE_M else 'b'} — what each editor actually generates: {mdl}",
                   f"fig4{'a' if mdl == BASE_M else 'b'}_editor_waterfall.png",
                   [tgt_cx], [gho_cx], ctx=CTX_EDITS, ylab="sample {s}")

---
## §4 — Is a successful edit's direction still invisible to the probe?

This is the measurement that turns "readable ≠ controllable" from an observation into a number.

Readout injection produces, by construction, $\Delta h_{\text{pinv}} = A^{+}(\text{target} - (Ah+b))$, which lies
**entirely in the row space of the probe**. It can never move `h` in any other direction. So the fraction of a
*successful* edit that lies in `row(A)`,

$$f \;=\; \frac{\lVert P_{\text{row}(A)}\,\Delta h\rVert}{\lVert \Delta h\rVert}, \qquad P_{\text{row}(A)} = A^{+}A$$

is not a descriptive statistic — it is the **hard ceiling on how much of a successful edit that editor could ever
achieve**, and equally the largest cosine any injection-style edit could reach with the truth.

**The chance level is mandatory.** The row space is `d` of `H` dimensions, so a *random* vector already scores
`√(d/H)` = 0.125 for the `d=4` position probe at `H=256`. Because `H512` has a different `H` (and hence chance
0.088), the figure plots **enrichment `f / chance`**; plotting the raw fraction across differing `H` would
manufacture a trend that is entirely the moving chance level.

In [ ]:
# [8] Table 5 + Fig 5 — row-space fraction of a successful Δh, per sample then averaged, against chance.
def row_frac(dh, P_row):
    num = torch.linalg.norm(dh @ P_row.T, dim=-1)
    den = torch.linalg.norm(dh, dim=-1).clamp_min(1e-9)
    return (num / den).cpu().numpy()

DH_KEYS = ["counterfactual state overwrite (oracle)", "freeze-time teacher forcing (oracle)",
           "decoder gradient (oracle)", "readout injection (structural)"]
RF = {}
rows = ["| model | Δh from | probe | row-space fraction `f` | chance `√(d/H)` | enrichment `f / chance` |",
        "|---|---|---|---|---|---|"]
for name in MODELS:
    RF[name] = {}
    Hdim = MODELS[name].hidden_size
    for dn in DH_KEYS:
        dh = STATES[name][dn] - STATES[name]["unsteered (no edit)"]
        RF[name][dn] = {}
        for pn, P in PROBES[name].items():
            f = row_frac(dh, P["P_row"]); chance = np.sqrt(P["d"] / Hdim)
            RF[name][dn][pn] = (float(f.mean()), float(f.std()), float(chance))
            rows.append(f"| {name} | {dn} | {pn} | **{f.mean():.3f}** ± {f.std():.3f} | {chance:.3f} | "
                        f"{f.mean()/chance:.2f}× |")
display(Markdown("**Table 5 — how much of a successful edit can a linear probe even see?** `f` is computed per "
                 "sample then averaged (± sd across samples), and is also the **ceiling on the cosine** any "
                 "injection-style editor could reach with that Δh. An enrichment near **1.00× is chance** — the "
                 "successful edit is, as far as the probe is concerned, indistinguishable from a random "
                 "direction. Readout injection is listed last as the sanity check: it is 1.000 by construction.\n\n"
                 + "\n".join(rows)))

fig, ax = plt.subplots(1, 2, figsize=(15.5, 4.8))
names = list(MODELS); oracle_keys = DH_KEYS[:3]; xi = np.arange(len(oracle_keys)); w = 0.15
for j, pn in enumerate(["position (d=4)", "position+velocity (d=8)"]):
    a = ax[j]
    for k, name in enumerate(names):
        a.barh(xi + (k - (len(names)-1)/2) * w, [RF[name][d][pn][0] / RF[name][d][pn][2] for d in oracle_keys],
               w, color=CLR[name], label=name)
    a.axvline(1.0, color="0.35", ls=":", lw=1.4)
    a.annotate("chance", xy=(1.0, len(oracle_keys) - 0.45), fontsize=7.5, color="0.35",
               ha="center", va="top", rotation=90)
    a.set_yticks(xi); a.set_yticklabels([k.replace(" (oracle)", "") for k in oracle_keys], fontsize=8)
    a.set_xlabel("enrichment  ‖P_row·Δh‖/‖Δh‖  ÷  chance √(d/H)")
    a.set_title(f"({'ab'[j]}) probe: {pn}", fontsize=10); a.grid(alpha=0.3, axis="x"); style_ax(a)
fig.legend(handles=[Line2D([0],[0], color=CLR[n], lw=6, label=n) for n in names],
           loc="upper center", ncol=3, fontsize=8.5, frameon=False, bbox_to_anchor=(0.5, 1.10))
fig.suptitle("Fig 5 — the reachability ceiling: what share of a SUCCESSFUL edit lies in the probe's row space, "
             "relative to what a random vector already has", y=1.17, fontsize=12)
fig.tight_layout(); fig.savefig(f"{OUT}/fig5_row_space.png", dpi=130, bbox_inches="tight")
display(fig); plt.close(fig)

---
## §5 — Superposition, with the affine identity broken by construction

**The question.** If the latent genuinely factored into objects, per-object edits would be independent
displacements that add:

$$[\,\text{move obj0}\,] \;+\; [\,\text{move obj1}\,] \;\overset{?}{=}\; [\,\text{move both}\,]$$

Concretely: build the counterfactual-overwrite state for "obj0 moved" (`h_A`), for "obj1 moved" (`h_B`), and for
"both moved" (`h_AB`), all from the same baseline `h_base`, then compare the **composed** displacement
`(h_A − h_base) + (h_B − h_base)` against the **direct** one `h_AB − h_base`.

Mechanism is **counterfactual overwrite** throughout — it is memoryless (it replaces the whole history), so it
isolates the *configuration → latent* map rather than a dynamics-mediated edit. All four states come from one
identical pipeline; the only thing that varies is the target configuration.

**Two identities that can fake this result, and why this notebook can finally separate them.**

**(i) An affine decoder.** With a single-`Linear` decoder, `decode(h_base + Δ_composed)` is *identically*
`decode(h_A) + decode(h_B) − decode(h_base)` — for any vectors at all. On the baseline this is exact to
`6.6e-08` (Table 1), so its composed Edit Index measures the decoder, not the representation. **The nonlinear
variants are not subject to this**, which is the whole point of the section: for them, the composed Edit Index
is a real measurement, and the question becomes whether it **beats what an affine decoder would have delivered
anyway**.

**(ii) An additive renderer.** In observation space, `obs(A moved) + obs(B moved) − obs(neither) = obs(both
moved)` is *also* an identity for non-overlapping objects — the `−obs(neither)` term is exactly what erases the
two stale objects. It breaks only where the objects **share rays** (the renderer takes the nearest hit, it does
not sum) or where the arithmetic leaves `[0,1]` and is clipped. This one is model-free, so it is measured
directly and quoted as the **ceiling a perfect model would score by obeying identity (ii) alone**.

**A note on the waterfall (Fig 7).** Counterfactual overwrite *fabricates* a history for every configuration,
so there is no single "real" context: the frames above the edit line are the **baseline (unedited) fabricated
history**, shown only to orient the reader, and the honest comparison is everything below the line — each
column's own free-run. The displayed samples are restricted to ones where both displaced objects stay in the
frustum for the whole rollout, because otherwise the GT column goes black within a few frames and shows nothing;
this affects only what is drawn, since every §5 number is scored at step 0 on the full sample set.

**What no identity can force.** The **state-space** comparison — `cos(composed, direct)` — relates two vectors
built by *independent* render → teacher-force runs. Decoder linearity says nothing about it. It carries two
empirical floors: a fully shuffled pairing, and the sharper "keep object A's true delta but take object B's from
a different sample", which isolates how much of the agreement needs the *correct pairing*.

In [ ]:
# [9] Build the four superposition configurations and their states, on every model.
XF, YF, YN, RAD_O = sim["x_far"], sim["y_far"], sim["y_near"], sim["radius"]
def in_frustum(p):
    x, y = p[..., 0], p[..., 1]
    return (y > YN + RAD_O) & (y < YF - RAD_O) & (np.abs(x) < (XF / YF) * y - RAD_O)

CIDX  = np.arange(COMP_N)
c_pre = edits.positions[CIDX, ef - 1, :N_OBJ, :].astype(np.float32)
c_vel = VEL_ALL[CIDX, ef, :N_OBJ, :].astype(np.float32)

def cf_history(targets):
    """Counterfactual history: EVERY object on a constant-velocity line arriving at its target at ef."""
    t_idx = np.arange(ef)
    hist = targets[:, None, :, :] - c_vel[:, None, :, :] * (ef - t_idx)[None, :, None, None] * DT
    return np.stack([render_traj(hist[i]) for i in range(len(hist))]).astype(np.float32)

def render_cfg_at(positions):
    return np.stack([render_traj(positions[i][None])[0] for i in range(len(positions))]).astype(np.float32)

def index_vs(pred, gt_edit, gt_uned):
    return _index_from(pred, gt_edit, gt_uned, np.abs(gt_edit - gt_uned) > 1e-3)

# one fixed displacement per object, so "move obj0" and "move obj1" are well-defined edits
delta0 = np.array([+2.0, +1.0], np.float32); delta1 = np.array([-2.0, +1.0], np.float32)
q = c_pre.copy(); q[:, 0] = c_pre[:, 0] + delta0; q[:, 1] = c_pre[:, 1] + delta1
ok_sup = in_frustum(q[:, 0]) & in_frustum(q[:, 1])

cfg_base = c_pre + c_vel * DT                       # the true continuation into frame ef (nothing edited)
cfg_A  = cfg_base.copy(); cfg_A[:, 0]  = q[:, 0]
cfg_B  = cfg_base.copy(); cfg_B[:, 1]  = q[:, 1]
cfg_AB = cfg_base.copy(); cfg_AB[:, 0] = q[:, 0]; cfg_AB[:, 1] = q[:, 1]
GT_BASE, GT_A, GT_B, GT_AB = (render_cfg_at(c) for c in (cfg_base, cfg_A, cfg_B, cfg_AB))
print(f"object-superposition samples: {ok_sup.sum()} of {COMP_N} (both moved objects in frustum); "
      f"displacements obj0 {tuple(delta0)}, obj1 {tuple(delta1)} sim units")

CF_OBS = {k: cf_history(c) for k, c in
          [("base", cfg_base), ("A", cfg_A), ("B", cfg_B), ("AB", cfg_AB)]}
COMP = {}
for name, m in MODELS.items():
    S = {k: warm(m, v, ef) for k, v in CF_OBS.items()}
    msk = ok_sup
    base, hA, hB, hAB = S["base"][msk], S["A"][msk], S["B"][msk], S["AB"][msk]
    COMP[name] = dict(base=base, h_A=hA, h_B=hB, h_AB=hAB,
                      d_direct=hAB - base, d_comp=(hA - base) + (hB - base))
print(f"built superposition states for {len(MODELS)} models on {int(ok_sup.sum())} samples")

In [ ]:
# [10] Table 6 + Fig 6 — superposition, and the control that separates a real result from a forced one.
ge, gu = GT_AB[ok_sup], GT_BASE[ok_sup]
g_shuf = torch.Generator().manual_seed(0)
perm = torch.randperm(int(ok_sup.sum()), generator=g_shuf)
def cs(a, b): return float(torch.nn.functional.cosine_similarity(a, b, dim=-1).mean())

g_rand = torch.Generator().manual_seed(7)

SUP = {}
for name, m in MODELS.items():
    v = COMP[name]; base, hA, hB = v["base"], v["h_A"], v["h_B"]
    dd, dc = v["d_direct"], v["d_comp"]
    # NULL 1 — wrong object B: keep object 0's true delta, take object 1's from a DIFFERENT sample.
    #   If composition is object-specific this must score far below the real composed state.
    d_wrongB = (hA - base) + (hB - base)[perm]
    # NULL 2 — a random direction of the SAME norm as d_comp. Tests the weakest possible claim:
    #   that adding any large vector to `base` produces a plausibly-edited scene.
    rnd = torch.randn(dc.shape, generator=g_rand).to(dc.device)
    d_rand = rnd / rnd.norm(dim=-1, keepdim=True) * dc.norm(dim=-1, keepdim=True)
    pred_comp   = roll(m, base + dc)[:, 0]                                        # what §5 scores
    pred_affine = roll(m, hA)[:, 0] + roll(m, hB)[:, 0] - roll(m, base)[:, 0]     # forced iff decode is affine
    SUP[name] = dict(
        cos=cs(dc, dd), angle=float(np.degrees(np.arccos(np.clip(cs(dc, dd), -1, 1)))),
        resid=float((torch.linalg.norm(dc - dd, dim=-1) / torch.linalg.norm(dd, dim=-1).clamp_min(1e-9)).mean()),
        mag=float((torch.linalg.norm(dc, dim=-1) / torch.linalg.norm(dd, dim=-1).clamp_min(1e-9)).mean()),
        cos_shuf_all=cs(dc[perm], dd), cos_shuf_B=cs((hA - base) + (hB - base)[perm], dd),
        idx_base=index_vs(roll(m, base)[:, 0], ge, gu), idx_direct=index_vs(roll(m, base + dd)[:, 0], ge, gu),
        idx_comp=index_vs(pred_comp, ge, gu), idx_affine=index_vs(pred_affine, ge, gu),
        idx_wrongB=index_vs(roll(m, base + d_wrongB)[:, 0], ge, gu),
        idx_rand=index_vs(roll(m, base + d_rand)[:, 0], ge, gu),
        roll_wrongB=roll(m, base + d_wrongB),
        affine_gap=float(np.sqrt(((pred_comp - pred_affine) ** 2).mean())),
        model_err=float(np.sqrt(((pred_comp - ge) ** 2).mean())),
        roll_comp=roll(m, base + dc), roll_direct=roll(m, base + dd), roll_base=roll(m, base),
        roll_A=roll(m, hA), roll_B=roll(m, hB))

rows = ["| model | `decode` | RMSE(composed decode, affine prediction) | Edit Index: unedited | random Δ, matched norm | composed w/ WRONG object B | **composed** | direct | what an affine decoder would force |",
        "|---|---|---|---|---|---|---|---|---|"]
for name in MODELS:
    s = SUP[name]
    rows.append(f"| {name} | {'affine' if MODELS[name].has_affine_decoder else 'nonlinear'} | "
                f"{s['affine_gap']:.2e} | {s['idx_base']:+.2f} | {s['idx_rand']:+.2f} | {s['idx_wrongB']:+.2f} | "
                f"**{s['idx_comp']:+.2f}** | {s['idx_direct']:+.2f} | {s['idx_affine']:+.2f} |")
id_pred = GT_A[ok_sup] + GT_B[ok_sup] - GT_BASE[ok_sup]
id_rmse = float(np.sqrt(((id_pred - ge) ** 2).mean()))
id_idx  = index_vs(id_pred, ge, gu)
overlap = float(np.mean(((GT_A[ok_sup] > 1e-3) & (GT_B[ok_sup] > 1e-3)).any(axis=-1)))
display(Markdown(
    "**Table 6a — does the composed state actually render the two-object edit?** Read left to right as a "
    "ladder: `unedited` is the floor, then two **null models** (a random displacement of the same size, and "
    "the same composition with object 1's delta taken from a different sample), then the real **composed** "
    "state, then the `direct` oracle it is trying to match. The last column is what an affine decoder would "
    "produce *by algebra*; it is **identical to `composed`** on the affine-decoder rows (which is why those "
    "rows are not evidence) and merely a comparison on the nonlinear rows. Note the affine column is not a "
    "null model — it already assumes each single edit works.\n\n"
    + "\n".join(rows) + "\n\n"
    f"**Model-free observation identity (ii).** `RMSE(GT_A + GT_B − GT_BASE, GT_AB)` = **{id_rmse:.4f}** against "
    f"an RMS signal of {float(np.sqrt((ge**2).mean())):.4f}; the two objects share at least one ray in "
    f"**{100*overlap:.0f}%** of samples, which is where the identity must break. Edit Index of that pure "
    f"observation identity: **{id_idx:+.3f}** — the ceiling a *perfect* model would score by obeying identity "
    f"(ii) alone."))

rows = ["| model | cos(composed, direct) | angle | relative residual ↓ | ‖composed‖/‖direct‖ | shuffled floor: another sample's composed | shuffled floor: object B from another sample |",
        "|---|---|---|---|---|---|---|"]
for name in MODELS:
    s = SUP[name]
    rows.append(f"| {name} | **{s['cos']:+.3f}** | {s['angle']:.0f}° | {s['resid']:.2f} | {s['mag']:.2f} | "
                f"{s['cos_shuf_all']:+.3f} | {s['cos_shuf_B']:+.3f} |")
display(Markdown("**Table 6b — the state-space comparison, which NO identity forces.** `d_composed` and "
                 "`d_direct` come from independent render → teacher-force runs. The shuffled columns are the "
                 "floors: the first for \"any edit delta resembles any other\", the second — the sharper one — "
                 "keeps object A's true delta and takes object B's from a different sample, so the margin above "
                 "it is what requires the *correct pairing*. Relative residual "
                 "`‖composed − direct‖/‖direct‖` is reported because it is the magnitude the cosine hides.\n\n"
                 + "\n".join(rows)))

fig, ax = plt.subplots(1, 3, figsize=(18.0, 4.8))
names = list(MODELS)
for k, name in enumerate(names):
    s = SUP[name]
    ax[0].bar(k, s["cos"], 0.6, color=CLR[name])
    ax[0].plot([k - 0.3, k + 0.3], [s["cos_shuf_B"]] * 2, color="0.2", lw=1.8)
    ax[0].plot([k - 0.3, k + 0.3], [s["cos_shuf_all"]] * 2, color="0.2", lw=1.8, ls=":")
ax[0].set_xticks([]); ax[0].set_ylabel("cos(composed, direct)"); ax[0].set_ylim(-0.05, 1.0)
ax[0].set_title("(a) do the displacements agree in state space?\nsolid line = wrong-object-B floor, "
                "dotted = fully shuffled floor", fontsize=9.5)
ax[0].grid(alpha=0.3, axis="y"); style_ax(ax[0])

xi = np.arange(5); w = 0.15
for k, name in enumerate(names):
    s = SUP[name]
    ax[1].bar(xi + (k - (len(names)-1)/2) * w,
              [s["idx_base"], s["idx_rand"], s["idx_wrongB"], s["idx_comp"], s["idx_direct"]], w, color=CLR[name])
ax[1].axhline(id_idx, color="0.25", ls="--", lw=1.4)
ax[1].annotate(f"ceiling from the render identity alone: {id_idx:+.2f}", xy=(4.45, id_idx), fontsize=7.5,
               color="0.25", ha="right", va="bottom")
ax[1].axhline(0, color="0.4", ls=":", lw=1.0)
ax[1].set_xticks(xi)
ax[1].set_xticklabels(["unedited", "random Δ\n(matched norm)", "composed w/\nWRONG object B",
                       "composed", "direct"], fontsize=7.5)
ax[1].set_ylabel("Edit Index"); ax[1].set_ylim(-1.05, 1.05)
ax[1].set_title("(b) does the composed state render the two-object edit?\nbars 2-3 are null models", fontsize=9.5)
ax[1].grid(alpha=0.3, axis="y"); style_ax(ax[1])

for k, name in enumerate(names):
    s = SUP[name]
    ax[2].bar(k, max(s["affine_gap"], 1e-9), 0.6, color=CLR[name])
ax[2].set_yscale("log"); ax[2].set_xticks([]); ax[2].set_ylabel("RMSE(composed decode, affine prediction)")
ax[2].set_title("(c) how far is this decoder from affine?\nat machine precision the composed index is "
                "forced by algebra", fontsize=9.5)
ax[2].grid(alpha=0.3, axis="y"); style_ax(ax[2])

fig.legend(handles=[Line2D([0],[0], color=CLR[n], lw=6, label=n) for n in names],
           loc="upper center", ncol=3, fontsize=8.5, frameon=False, bbox_to_anchor=(0.5, 1.12))
fig.suptitle("Fig 6 — object superposition under counterfactual overwrite, and the affine-decoder control",
             y=1.20, fontsize=12)
fig.tight_layout(); fig.savefig(f"{OUT}/fig6_superposition.png", dpi=130, bbox_inches="tight")
display(fig); plt.close(fig)

In [ ]:
# [11] Fig 7 — observation waterfalls for the two-object (superposition) edit.
def render_cfg_ids(positions):
    """Single-frame clean render of a configuration, returning intensity AND per-ray object id."""
    ints, ids = [], []
    for i in range(len(positions)):
        sc = Scene(positions=positions[i][None], velocities=np.zeros((1, N_OBJ, 2), np.float32),
                   radii=RAD, colors=COL, reflectivities=REFL, config=_cfg(1, 0.0))
        _, rid, rint = render_scene(sc); ints.append(rint[0]); ids.append(rid[0])
    return np.stack(ints).astype(np.float32), np.stack(ids).astype(np.int64)

# GT for the both-moved world, rolled FORWARD K steps (each object continues with its own velocity)
gt_ab_traj = np.zeros((COMP_N, K_ROLL, R), np.float32)
for s_ in range(K_ROLL):
    gt_ab_traj[:, s_] = render_cfg_ids(cfg_AB + c_vel * DT * s_)[0]
_, ids_AB   = render_cfg_ids(cfg_AB)
_, ids_BASE = render_cfg_ids(cfg_base)
tgt_cx2 = [np.array([_cx(ids_AB[i] == k) for i in range(COMP_N)]) for k in (0, 1)]
gho_cx2 = [np.array([_cx((ids_BASE[i] == k) & (ids_AB[i] != k)) for i in range(COMP_N)]) for k in (0, 1)]

# The counterfactual mechanism FABRICATES a history for every configuration, so there is no single
# "real" context. We show the baseline (unedited) counterfactual history purely to orient the reader;
# the honest comparison is everything below the edit line, each column's own free-run.
sup_ids = np.where(ok_sup)[0]
CTX_CF = CF_OBS["base"][:, ef - N_CTX:ef, :]

# Which samples are worth DISPLAYING. With both objects displaced and then rolled forward, some leave
# the frustum within a couple of frames; the GT column then goes black and the visual comparison is
# vacuous. Restrict the displayed samples to those where both moved objects stay in frustum for the
# whole displayed rollout. This changes only what is drawn — every number in §5 is scored at step 0
# and uses the full `ok_sup` set. Selection within the eligible pool is random, not by effect size.
roll_pos = cfg_AB[:, None, :, :] + c_vel[:, None, :, :] * DT * np.arange(K_ROLL)[None, :, None, None]
vis_roll = in_frustum(roll_pos).all(axis=(1, 2))
show_pool = np.where(ok_sup & vis_roll)[0]
rng_w = np.random.default_rng(0)
SHOW = [int(i) for i in rng_w.choice(show_pool, size=3, replace=False)]
print(f"displayable samples (both moved objects in frustum for all {K_ROLL} rollout steps): "
      f"{len(show_pool)} of {int(ok_sup.sum())} scored samples | showing {SHOW}")

def _expand(arr_masked):
    """Map an array indexed over the ok_sup subset back onto full COMP_N indexing for the plotter."""
    full = np.zeros((COMP_N,) + arr_masked.shape[1:], arr_masked.dtype)
    full[sup_ids] = arr_masked
    return full

for mdl in (BASE_M, MAIN_M, "nonlinear dec only · H256 · seed 0"):
    s = SUP[mdl]
    titles = ["GT (sim)\nboth objects moved",
              f"unedited (no edit)\nEdit Index {s['idx_base']:+.2f}",
              "move obj0 only", "move obj1 only",
              f"DIRECT: move both\nEdit Index {s['idx_direct']:+.2f}",
              f"COMPOSED: Δ(obj0) + Δ(obj1)\nEdit Index {s['idx_comp']:+.2f}",
              f"NULL: Δ(obj0) + Δ(obj1) from\nANOTHER sample · {s['idx_wrongB']:+.2f}"]
    bodies = [gt_ab_traj] + [_expand(b) for b in
              (s["roll_base"], s["roll_A"], s["roll_B"], s["roll_direct"], s["roll_comp"], s["roll_wrongB"])]
    tag = {BASE_M: "a", MAIN_M: "b"}.get(mdl, "c")
    waterfall_grid(titles, bodies, SHOW,
                   f"Fig 7{tag} — the two-object superposition edit: {mdl}",
                   f"fig7{tag}_superposition_waterfall.png",
                   tgt_cx2, gho_cx2, ctx=CTX_CF, ylab="sample {s}")

---
## §6 — Summary

Computed, not asserted: every line below is printed from the numbers above.

In [ ]:
# [12] Computed summary.
print("=" * 100)
print("NONLINEAR GRU — DO THE EDITABILITY FINDINGS SURVIVE A NONLINEAR READ-IN AND READ-OUT?")
print("=" * 100)
for name in MODELS:
    m = MODELS[name]; q = PQ[name]; s = SUP[name]
    u  = CARDS[name]["unsteered (no edit)"]["edit_index"]
    ri = CARDS[name]["readout injection (structural)"]["edit_index"]
    cf = CARDS[name]["counterfactual state overwrite (oracle)"]["edit_index"]
    ftm = CARDS[name]["freeze-time teacher forcing (oracle)"]["edit_index"]
    dg = CARDS[name]["decoder gradient (oracle)"]["edit_index"]
    lp = PROBES[name]["position (d=4)"]["r2"]; mp = MLPP[name]["position (d=4)"]["r2"]
    rf = RF[name]["counterfactual state overwrite (oracle)"]["position (d=4)"]
    print(f"\n--- {name}  [decode {'AFFINE' if m.has_affine_decoder else 'NONLINEAR'}] ---")
    print(f"  0. WORLD MODEL QUALITY: next-step RMSE vs clean {q['next_step']:.4f} "
          f"(baseline {PQ[BASE_M]['next_step']:.4f}) | open-loop K=15 {q['open_loop']:.4f} | "
          f"sharpness {q['tv_ratio']:.2f}")
    print(f"  1. LINEAR READABILITY of position: R2 {lp:.3f} (MLP {mp:.3f}, gap {mp-lp:+.3f}) -> "
          + ("still LINEARLY readable" if lp > 0.7 else "NO LONGER linearly readable"))
    print(f"  2. EDITABILITY BRACKET (Edit Index, this model's unsteered floor {u:+.2f}):")
    print(f"       structural  readout injection {ri:+.2f}   ({'INERT' if ri < u + 0.15 else 'MOVED'})")
    print(f"       oracles     counterfactual {cf:+.2f} | freeze-time {ftm:+.2f} | decoder gradient {dg:+.2f}")
    print(f"       -> {'oracles succeed while the structural editor is inert: the barrier is the EDIT MAP, not the model' if (cf > u + 0.5 and ri < u + 0.15) else 'bracket NOT reproduced — inspect Table 4'}")
    print(f"  3. REACHABILITY CEILING (position probe, chance {rf[2]:.3f}): f = {rf[0]:.3f} "
          f"({rf[0]/rf[2]:.2f}x chance) -> a readout-injection edit could match at best "
          f"{100*rf[0]:.0f}% of a successful edit's direction")
    print(f"  4. SUPERPOSITION, decoded observation: composed {s['idx_comp']:+.2f} vs "
          f"affine-forced {s['idx_affine']:+.2f} vs direct {s['idx_direct']:+.2f}; "
          f"render-identity ceiling {id_idx:+.2f}")
    print(f"       decoder distance from affine: {s['affine_gap']:.2e} -> "
          + ("FORCED BY ALGEBRA, carries no information about the latent"
             if s["affine_gap"] < 1e-5 else
             ("real measurement, and composed BEATS the affine baseline" if s["idx_comp"] > s["idx_affine"] + 0.02
              else "real measurement, but composed does NOT beat the affine baseline")))
    print(f"  5. SUPERPOSITION, state space (nothing forces this): cos {s['cos']:+.3f} ({s['angle']:.0f}deg) "
          f"vs floors {s['cos_shuf_B']:+.3f} (wrong object B) / {s['cos_shuf_all']:+.3f} (fully shuffled); "
          f"relative residual {s['resid']:.2f}")
print("\n" + "=" * 100)
print("figures written to", os.path.abspath(OUT), ":", sorted(os.listdir(OUT)))

### Current results (updated 2026-08-05)

*This block is the only place in the notebook where results live; every section above states what is measured,
not what it currently reads. All numbers from the run above: `4_fixed_refl_inview`, N=256 edits (§3–§4),
67 in-frustum samples (§5).*

**Headline: every main finding survives a nonlinear encoder and decoder — including object superposition, which
the nonlinear models turn from an unfalsifiable number into a real, object-specific result. What does not
survive is the *linear* models' decode-level evidence for it, which was forced by algebra.**

**§1 — no quality confound.** The nonlinear variants are, if anything, *slightly better* world models than the
baseline: next-step RMSE vs clean **0.1029–0.1033** against the baseline's **0.1041**, open-loop K=15
**0.150–0.154** against **0.155**, and closer to the GT's sharpness (TV ratio **1.14** vs **1.29**). Nothing
below can be attributed to a degraded model. All five unsteered Edit Indices agree to 0.01 (−0.67/−0.68), so the
scale's `−1` end sits in the same place for every model.

**§2 — position is still linearly readable; encoder depth did not hide it.** Linear position R² is
**0.803–0.805** on the nonlinear enc+dec runs versus **0.815** on the baseline (held-out, like-for-like). The
MLP−linear gap widens only slightly (**+0.083 → +0.102**). The "readable" half of *readable ≠ controllable* is
unaffected, so §4's row-space measurement means the same thing it did before.

**§3 — editability still fails, with the bracket intact on every model.** Readout injection is **inert**:
**−0.65** against its own unsteered **−0.67/−0.68**, on all five models — visually indistinguishable from the
unsteered column in Fig 4. Meanwhile counterfactual overwrite reaches **+0.68 on every single model** and
freeze-time **+0.52…+0.59**. Same model, same decoder, same rollout. The barrier is the **reachability of the
edit map**, and it is not a shallow-read-out artifact.

> **One genuinely new result.** The **decoder-gradient oracle weakens sharply on a nonlinear decoder**: **+0.97**
> (baseline) and **+1.00** (H512) drop to **+0.68…+0.72**. This is expected and worth stating — against an affine
> decoder that oracle solves a *convex* least-squares problem in `h` and can essentially always hit the target,
> whereas through an MLP it descends a nonconvex objective and no longer reaches it. Its near-perfect score on
> the linear models was partly a property of the decoder's convexity, not evidence that the state was that
> reachable. It is still an oracle and still clears the ghost (Ghost RMSE 0.06–0.08 vs 0.56–0.59 unsteered).

**§4 — a successful edit is still invisible to the probe.** The row-space fraction of the counterfactual Δh is
**0.131–0.155** on the nonlinear models against a chance level of **0.125** — an enrichment of
**1.05×…1.24×**, i.e. still essentially chance. The baseline sits *below* chance (0.096, **0.76×**). So the
nonlinear models are marginally less orthogonal to the probe, but nowhere near enough to matter: a
readout-injection edit could still match at most **13–15%** of a successful edit's direction.

**§5 — the *decode-level evidence* on the linear models was an artifact; on the nonlinear models
compositionality is real, object-specific, and holds.** These are two different statements and the earlier
draft of this block conflated them.

- **What was an artifact.** On both affine-decoder models the composed decode equals the affine prediction to
  **6.6e-08 / 9.0e-08** — machine precision. Their composed Edit Index (**+0.46**) is *algebraically determined*,
  so it was never evidence about the latent, whatever its value. It also sits at 90% of the **+0.51** ceiling the
  model-free render identity scores on its own.
- **What the nonlinear models establish.** Their decoders depart from affine by **8.5e-02…8.8e-02** (roughly half
  their own total error), so the composed index is a genuine measurement — and against the proper null models it
  is decisive:

  | | unedited | random Δ, matched norm | composed w/ **wrong** object B | **composed** | direct |
  |---|---|---|---|---|---|
  | nonlinear enc+dec · seed 0 | −0.74 | −0.46 | −0.18 | **+0.44** | +0.72 |
  | nonlinear enc+dec · seed 1 | −0.74 | −0.46 | −0.20 | **+0.43** | +0.71 |
  | nonlinear dec only · seed 0 | −0.74 | −0.45 | −0.19 | **+0.43** | +0.72 |

  Summing two independently-built per-object displacements yields a state that renders **both** objects at
  approximately the right places — **+0.44 against a −0.18 wrong-object-B null and a −0.46 random-Δ null**.
  Swapping in another sample's object-1 delta collapses the result to the unedited side, so the composition is
  **object-specific**, not a generic "any large displacement looks edited" effect. Fig 7's rightmost column shows
  the same thing in observation space. **This is a stronger result than the linear models could support**, because
  there the identical number was unfalsifiable.
- **The affine column is not a null model.** It already presupposes that each single-object edit works, so
  "composed ≈ affine" says the nonlinear decoder behaves near-affinely along these particular directions — it
  does **not** cancel the finding. The genuine nulls are the two above.
- **It is partial, not exact.** Composed (**+0.43…+0.44**) recovers ~82% of the index gain from unedited to
  `direct` (**+0.72**), and lands slightly below the **+0.51** render-identity ceiling. So compositionality holds
  substantially but imperfectly, and the composed state is measurably worse than the true both-moved state.
  Fig 7 sample 47 shows the shortfall concretely: the COMPOSED column develops banding where DIRECT stays clean.
- **In state space it is real on every model but *weaker* with a nonlinear read-out path**, which is the opposite
  of a "shallow decoder was hiding the structure" story: `cos(composed, direct)` falls from **+0.873 (29°)** to
  **+0.784…+0.801 (37–38°)**, relative residual **0.52 → 0.74–0.78**, against floors of **+0.31…+0.37**
  (wrong-object-B) and **~+0.05** (fully shuffled).

**What this thread changes about the thread it belongs to.** The editability negative is now known to be
independent of the read-in/read-out nonlinearity as well as of capacity (H=8…512), observation noise, action
training, and architecture (GRU/RSSM). `delta_h_analysis` §7's *decode-level* superposition numbers are **not evidence on
that (affine-decoder) model** and must not be cited without its §7b control; the claim they were making is,
however, **independently confirmed here** on decoders where it can actually be tested. Its state-space claim
stands on every model, with the floors above as its proper reference.

**Caveats.** One dataset, two objects, one displacement pair for §5, `N=67` scored superposition samples. The
nonlinear variants carry two seeds; the baselines carry one. Depth was fixed at 2 blocks and the activation at
ReLU — no depth or activation sweep was run.